# Embeddings

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encode the query vector.

In [6]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

Encode the document vector.

In [7]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

Compare how similar they are to each other. The `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with unit length. When both vectors are normalized, the dot product equals cosine similarity. 

In [8]:
v1.dot(dv)

np.float32(0.32332402)

Try an unrelated query.

In [10]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
v2.dot(dv)

np.float32(0.01973048)

Higher score means higher similarity.

# Embedding the dataset

Use module from previous mini-project to load the FAQ data.

In [11]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-29 11:11:08--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     888  --.-KB/s    in 0s      

2026-06-29 11:11:09 (13.9 MB/s) - ‘ingest.py’ saved [888/888]



In [18]:
from ingest import load_faq_data
from tqdm.auto import tqdm
import numpy as np

In [21]:
documents = load_faq_data()
documents[:3]

[{'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.",
  'doc_id': '9e508f2212'},
 {'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\

Generate embeddings.

In [13]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

Chunk the dataset into batches of 50 and encode each batch.

In [15]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

Turn them into a 2-dimensional array (matrix) where rows are documents (vectors) and columns are dimensions of the vectors.

In [20]:
X = np.array(vectors)
X.shape

(1350, 384)

# Vector search

Scoring documents.

In [22]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [23]:
scores = X.dot(v_query)

Best match.

In [24]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629411))

In [25]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

Top 5 results.

In [26]:
top5 = np.argsort(scores)[-5:]

In [27]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [28]:
scores[top5]

array([0.7629411 , 0.7579372 , 0.7192134 , 0.65363127, 0.5601    ],
      dtype=float32)

Or, equivalently.

In [29]:
top5 = np.argsort(-scores)[:5]

Print top 5 documents.

In [30]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629411
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192134
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions',

# Vector search with `minsearch`

Creating the index.

In [31]:
from minsearch import VectorSearch

In [32]:
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

Searching.

In [33]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [34]:
results[0]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'doc_id': '74eb249bbf'}

Filtering by course.

In [35]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

# RAG with vector search

Pull RAG logic from previous project.

In [42]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-29 11:41:34--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.3’

rag_helper.py.3     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-29 11:41:35 (5.22 MB/s) - ‘rag_helper.py.3’ saved [2134/2134]



In [41]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI
from ingest import load_faq_data, build_index

Create the OpenAI client.

In [45]:
load_dotenv()
openai_client = OpenAI()

Download and index the data.

In [43]:
documents = load_faq_data()
index = build_index(documents)

In [46]:
assistant = RAGBase(
    index = index,
    llm_client = openai_client,
)

In [ ]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

Ask it a question.

In [47]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while they’re still accepting submissions.'

That still uses keyword search. We need to override `search` in our RAG logic.

In [48]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

Now use it.

In [49]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [50]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. If you want a certificate, make sure you submit your project while submissions are still open.'